# Demo 05 - backfill files into bronze

Job task. Watches the landing zone for device exports and appends them to the same
`sensor_readings_bronze` the Event Hub writes to, tagged `source_system = file`.

This is where schema evolution happens. `addNewColumns` is the default and it behaves the way it
does on purpose: an unknown column stops the stream, the wider schema is written to the schema
location, and the next run continues from it. In a job with retries enabled that shows up as one
failed attempt followed by a green one, with nobody editing any code in between.

The alternative is `rescue`, where the stream never stops and unknown fields land in
`_rescued_data` instead of becoming columns. Uptime instead of clean columns, pick one.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"])

login   = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("target_catalog")
verbose = dbutils.widgets.get("verbose") == "true"
assert all([login, catalog])

demo    = f"{login}_demo_bronze"
source  = f"/Volumes/{catalog}/{demo}/demo_landing/readings"
target  = f"{catalog}.{demo}.sensor_readings_bronze"
ckpt    = f"/Volumes/{catalog}/{demo}/checkpoints/backfill"

print(source, "->", target)

In [0]:
def has_files(path):
    # Auto Loader cannot infer a schema from an empty folder, and on the first job run
    # there is legitimately nothing here yet
    try:
        return len(dbutils.fs.ls(path)) > 0
    except Exception:
        return False


if not has_files(source):
    print("landing zone empty, nothing to backfill")
    dbutils.notebook.exit("no files")

In [0]:
q = (spark.readStream.format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.schemaLocation", f"{ckpt}/schema")
     .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
     .option("rescuedDataColumn", "_rescued_data")
     .option("cloudFiles.maxFilesPerTrigger", 10)
     .load(source)
     .withColumn("source_system", F.lit("file"))
     .withColumn("source_file",   F.col("_metadata.file_path"))
     .withColumn("ingestion_ts",  F.current_timestamp())
     .withColumn("load_date",     F.current_date())
     .writeStream
     .option("checkpointLocation", ckpt)
     .option("mergeSchema", "true")
     .trigger(availableNow=True)
     .toTable(target))
q.awaitTermination()

print(target, "->", spark.table(target).count(), "rows total")

In [0]:
# one row per device and source, so it is obvious which path each device came in on.
# psu_temp_c and fan_rpm only exist once the part 2 file has been through, hence the check
if verbose:
    cols  = spark.table(target).columns
    extra = [c for c in ("psu_temp_c", "fan_rpm") if c in cols]

    aggs = [F.count("*").alias("readings")]
    aggs += [F.count(c).alias(f"with_{c}") for c in extra]

    print("new columns so far:", extra or "none yet")
    display(spark.table(target)
            .groupBy("device_id", "source_system")
            .agg(*aggs)
            .orderBy("device_id", "source_system"))

In [0]:
# the schema on disk is versioned, one file per version
if verbose:
    display(dbutils.fs.ls(f"{ckpt}/schema/_schemas"))